# Click-Assisted Labeling: 1 Click -> Whole Sequence (SAM3 Tracker Propagation)

**For the classes where text-prompted SAM3 fails** (woodenboard 17%, dummychild 33%,
bicyclestand 50% — see `docs/low_object_detection_plan.md`, Phase B gate result).

**Why one click is enough per sequence:** the target object is static and the car
approaches it. You click the object once on a mid-approach frame; the SAM3 tracker
(the same `facebook/sam3` checkpoint used for text prompting, in video point-prompt
mode) propagates the mask across the whole approach, so every usable frame gets a box
from that single click (~15-30 labeled frames per click).

**Workflow (designed for ~100-clicks sessions, resumable across Colab disconnects):**
1. Select sequences per failing class from the inventory.
2. Extract approach frames (ratio 0.30-0.85) per sequence.
3. Click round: one image per sequence, click the object (or press Skip).
   Clicks are saved to JSON after every click — reconnecting resumes where you left off.
4. Propagation round (GPU): the SAM3 tracker follows the clicked object through all frames.
5. Export: YOLO-format boxes + QA overlays + per-sequence stats.

Clicked+propagated masks double as the **gold QA set** for measuring the text-prompt
pseudo-label precision later.

**Requires a GPU runtime** for step 4 (clicking itself works on CPU).

## 1. Setup

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if IN_COLAB:
    # SAM3 (incl. the tracker) ships in recent transformers releases.
    pip_install("-U", "transformers", "accelerate")

import base64
import json
import random
import re
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"colab={IN_COLAB} device={DEVICE}")

## 2. Config

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def find_repo_root() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    colab_clone = Path("/content/vision-uss-research")
    return colab_clone if colab_clone.exists() else None

PROFILE = "colab_drive" if IN_COLAB else "local"
RAW_ROOT_OVERRIDE = None
INVENTORY_CSV_OVERRIDE = None
OUTPUT_DIR_OVERRIDE = None      # keep on Drive so clicks survive disconnects

repo_root = find_repo_root()
if repo_root:
    sys.path.insert(0, str(repo_root / "src"))
    from vision_uss_research.settings import load_paths

    paths = load_paths(profile=PROFILE)
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
    INVENTORY_CSV = Path(
        INVENTORY_CSV_OVERRIDE
        or paths.outputs_dir / "profiles" / "object_inventory" / "inventory_folders.csv"
    )
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE or paths.outputs_dir / "experiments" / "click_labels")
else:
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE)
    INVENTORY_CSV = Path(INVENTORY_CSV_OVERRIDE)
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLICKS_JSON = OUTPUT_DIR / "clicks.json"          # resumable click store
FRAMES_ROOT = OUTPUT_DIR / "frames"               # extracted approach frames
LABELS_ROOT = OUTPUT_DIR / "labels"               # YOLO txt output
OVERLAYS_ROOT = OUTPUT_DIR / "overlays"           # QA overlays

# --- session knobs ----------------------------------------------------------
TARGET_OBJECTS = {"woodenboard": 100, "dummychild": 35, "bicyclestand": 35}  # clicks per class
MANUAL_SEQUENCE_IDS: list[str] = []
CLICK_RATIO = 0.60            # frame shown for clicking
RESET_CLICK_HISTORY = False   # True: archive clicks.json and start the click round
                              # fresh (old answers kept as clicks_backup_*.json)
# Thin/extended structures: one point selects a single bar, so prompt with a 2-corner
# BOX instead (click top-left-ish corner, then bottom-right-ish corner of the stand).
BOX_PROMPT_OBJECTS = {"bicyclestand"}
# Re-annotate only these objects: their existing clicks are re-asked and their
# propagated rows dropped, without touching other objects' finished work.
REDO_OBJECTS: set[str] = set()   # e.g. {"bicyclestand"}
RATIO_RANGE = (0.30, 0.85)    # approach window to extract/propagate over
N_FRAMES = 20                 # frames extracted per sequence inside the window
SEED = 0
SAM3_MODEL_ID = "facebook/sam3"
CLASS_IDS = {"woodenboard": 0, "dummychild": 1, "bicyclestand": 2}  # YOLO class ids

print(f"raw root : {RAW_ROOT}")
print(f"outputs  : {OUTPUT_DIR}")

## 3. Select sequences

In [ ]:
inv = pd.read_csv(INVENTORY_CSV)
usable = inv[
    (inv["status"] == "complete")
    & ((inv["has_front_video"] == 1) | (inv["has_rear_video"] == 1))
].copy()

rng = random.Random(SEED)
selected: list[dict] = []

if MANUAL_SEQUENCE_IDS:
    for seq_id in MANUAL_SEQUENCE_IDS:
        row = inv[inv["sequence_id"] == seq_id].iloc[0]
        objs = str(row["label_v2_objects"]).split("|")
        target = next((o for o in objs if o.lower() != "empty"), objs[0])
        selected.append({"sequence_id": seq_id, "target_object": target})
else:
    for obj, n_wanted in TARGET_OBJECTS.items():
        mask = usable["label_v2_objects"].fillna("").str.split("|").apply(
            lambda names: any(n.strip().lower() == obj for n in names)
        )
        candidates = sorted(usable[mask]["sequence_id"].tolist())
        picks = rng.sample(candidates, min(n_wanted, len(candidates)))
        selected += [{"sequence_id": s, "target_object": obj} for s in picks]
        print(f"{obj:<15} {len(candidates):>5} candidates -> selected {len(picks)}")

selected_df = pd.DataFrame(selected)
print(f"\ntotal sequences to click: {len(selected_df)}")

## 4. Extract approach frames (once per sequence, cached)

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".webm", ".avi"}

def sequence_files(seq_dir: Path) -> dict:
    top = [p for p in seq_dir.iterdir() if p.is_file()]
    def pick(pred):
        matches = sorted(p for p in top if pred(p))
        return matches[0] if matches else None
    return {
        "metadata_json": pick(lambda p: p.suffix.lower() == ".json"),
        "front_video": pick(lambda p: "front" in p.name.lower() and p.suffix.lower() in VIDEO_EXTENSIONS),
        "rear_video": pick(lambda p: "rear" in p.name.lower() and p.suffix.lower() in VIDEO_EXTENSIONS),
    }

def select_camera_videos(seq_dir: Path) -> list[tuple[str, Path]]:
    found = sequence_files(seq_dir)
    direction = "unknown"
    if found["metadata_json"]:
        with found["metadata_json"].open() as f:
            metadata = json.load(f)
        for tag in metadata.get("tags", []):
            if tag.get("tagSubCategory", {}).get("name", "").lower() == "driving direction":
                direction = tag.get("name", "unknown")
    if direction == "backward":
        videos = [("rear", found["rear_video"])]
    elif direction == "forward":
        videos = [("front", found["front_video"])]
    else:
        videos = [("front", found["front_video"]), ("rear", found["rear_video"])]
    return [(c, p) for c, p in videos if p is not None]

def extract_sequence_frames(seq_id: str, camera: str, video_path: Path) -> Path | None:
    """Extract N_FRAMES JPEGs inside RATIO_RANGE into frames/<seq>/<camera>/ (cached).
    The tracker consumes an ordered frame list; JPEGs named 00000.jpg, 00001.jpg..."""
    out_dir = FRAMES_ROOT / seq_id / camera
    if out_dir.exists() and len(list(out_dir.glob("*.jpg"))) >= N_FRAMES - 2:
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    duration_ms = 1000.0 * cap.get(cv2.CAP_PROP_FRAME_COUNT) / max(cap.get(cv2.CAP_PROP_FPS), 1e-6)
    ratios = np.linspace(RATIO_RANGE[0], RATIO_RANGE[1], N_FRAMES)
    kept = 0
    for i, r in enumerate(ratios):
        cap.set(cv2.CAP_PROP_POS_MSEC, r * duration_ms)
        ok, frame = cap.read()
        if ok:
            cv2.imwrite(str(out_dir / f"{i:05d}.jpg"), frame, [cv2.IMWRITE_JPEG_QUALITY, 92])
            kept += 1
    cap.release()
    return out_dir if kept else None

import time

def with_drive_retry(fn, *args, retries=2, wait_s=3):
    """Colab's Drive FUSE mount throws transient OSError (Errno 5) - retry, then give up."""
    for attempt in range(retries + 1):
        try:
            return fn(*args)
        except OSError:
            if attempt == retries:
                raise
            time.sleep(wait_s)

# index of what to click: one (sequence, camera, click frame) per row
click_items = []
skipped: list[dict] = []

for _, sel in selected_df.iterrows():
    seq_dir = RAW_ROOT / sel["sequence_id"]
    try:
        if not with_drive_retry(seq_dir.is_dir):
            skipped.append({"sequence_id": sel["sequence_id"], "error": "not_a_dir"})
            continue
        videos = with_drive_retry(select_camera_videos, seq_dir)
    except Exception as e:
        skipped.append({"sequence_id": sel["sequence_id"], "error": f"{type(e).__name__}: {e}"})
        continue

    for camera, video_path in videos:
        try:
            frames_dir = with_drive_retry(
                extract_sequence_frames, sel["sequence_id"], camera, video_path
            )
        except Exception as e:
            skipped.append({"sequence_id": sel["sequence_id"], "camera": camera,
                            "error": f"{type(e).__name__}: {e}"})
            continue
        if frames_dir is None:
            continue
        click_frame_idx = int(round(
            (CLICK_RATIO - RATIO_RANGE[0]) / (RATIO_RANGE[1] - RATIO_RANGE[0]) * (N_FRAMES - 1)
        ))
        click_items.append({
            "sequence_id": sel["sequence_id"],
            "target_object": sel["target_object"],
            "camera": camera,
            "frames_dir": str(frames_dir),
            "click_frame_idx": click_frame_idx,
        })

print(f"click items (sequence x camera): {len(click_items)}")
if skipped:
    skipped_df = pd.DataFrame(skipped)
    skipped_df.to_csv(OUTPUT_DIR / "skipped_sequences.csv", index=False)
    print(f"skipped {len(skipped)} (Drive I/O errors or broken folders) "
          f"-> {OUTPUT_DIR / 'skipped_sequences.csv'}")
    display(skipped_df)

## 5. Click round

One image per item. **Click the target object** (the click turns green), or press the
Skip button if the object is not visible in this camera/frame. Progress is saved to
`clicks.json` after every answer — rerun this cell any time to resume.

When direction was unknown both cameras are shown; skip the one without the object.

In [ ]:
import time

if RESET_CLICK_HISTORY and CLICKS_JSON.exists():
    backup = CLICKS_JSON.with_name(f"clicks_backup_{time.strftime('%Y%m%d_%H%M%S')}.json")
    CLICKS_JSON.rename(backup)
    print(f"click history archived to {backup.name} - starting fresh\n")

def load_clicks() -> dict:
    if CLICKS_JSON.exists():
        return json.loads(CLICKS_JSON.read_text())
    return {}

def resolve_click_image(item: dict) -> Path | None:
    """The intended click frame can be missing when extraction was partial (corrupt
    video segment, Drive glitch). Fall back to the nearest existing frame; None if the
    folder has no frames at all."""
    frames_dir = Path(item["frames_dir"])
    preferred = frames_dir / f"{item['click_frame_idx']:05d}.jpg"
    if preferred.exists():
        return preferred
    available = sorted(frames_dir.glob("*.jpg"))
    if not available:
        return None
    return min(available, key=lambda p: abs(int(p.stem) - item["click_frame_idx"]))

def save_clicks(clicks: dict) -> None:
    CLICKS_JSON.write_text(json.dumps(clicks, indent=1))

def item_key(item: dict) -> str:
    return f"{item['sequence_id']}::{item['camera']}"

def ask_click_colab(image_path: Path, title: str, mode: str = "point"):
    """mode='point': click the object (re-click to adjust), Confirm returns (x, y).
    mode='box': click two opposite corners; a dashed rectangle previews the box;
    another click restarts. Confirm returns (x0, y0, x1, y1). Skip returns None."""
    from google.colab.output import eval_js
    from IPython.display import display, HTML

    b64 = base64.b64encode(image_path.read_bytes()).decode()
    uid = f"clk_{random.randrange(10**9)}"
    hint = ("draw a BOX: click one corner of the object"
            if mode == "box" else "click the object...")
    display(HTML(f'''
      <div style="font-family:sans-serif;margin:4px 0"><b>{title}</b></div>
      <div style="position:relative;display:inline-block;line-height:0">
        <img id="{uid}" src="data:image/jpeg;base64,{b64}"
             style="max-width:100%;cursor:crosshair;border:2px solid #888"/>
        <div id="{uid}_mk" style="position:absolute;display:none;width:26px;height:26px;
             margin:-13px 0 0 -13px;pointer-events:none;
             border:3px solid lime;border-radius:50%;
             box-shadow:0 0 0 2px rgba(0,0,0,.6)">
          <div style="position:absolute;left:50%;top:50%;width:4px;height:4px;
               margin:-2px 0 0 -2px;background:red;border-radius:50%"></div>
        </div>
        <div id="{uid}_rect" style="position:absolute;display:none;pointer-events:none;
             border:3px dashed lime;box-shadow:0 0 0 1px rgba(0,0,0,.6)"></div>
      </div>
      <div style="display:flex;align-items:center;gap:14px;margin:6px 0">
        <div style="text-align:center;font-family:sans-serif;font-size:12px;color:#555">
          <canvas id="{uid}_zoom" width="180" height="180"
                  style="border:2px solid #888;background:#222;display:block"></canvas>
          zoomed check
        </div>
        <div>
          <div id="{uid}_st" style="font-family:sans-serif;margin:6px 0;color:#555">
            {hint}</div>
          <button id="{uid}_ok" disabled
                  style="padding:6px 20px;font-weight:bold">Confirm</button>
          <button id="{uid}_skip" style="padding:6px 16px;margin-left:8px">Skip</button>
        </div>
      </div>
    '''))
    coords = eval_js(f'''
      new Promise(resolve => {{
        const MODE = "{mode}";
        const img = document.getElementById("{uid}");
        const mk = document.getElementById("{uid}_mk");
        const rect = document.getElementById("{uid}_rect");
        const st = document.getElementById("{uid}_st");
        const ok = document.getElementById("{uid}_ok");
        const zoom = document.getElementById("{uid}_zoom");
        let pt = null, c1 = null, box = null;
        const drawZoom = (nx, ny) => {{
          const half = 30;
          const ctx = zoom.getContext("2d");
          ctx.imageSmoothingEnabled = false;
          ctx.clearRect(0, 0, zoom.width, zoom.height);
          ctx.drawImage(img, nx - half, ny - half, 2 * half, 2 * half,
                        0, 0, zoom.width, zoom.height);
          ctx.strokeStyle = "lime"; ctx.lineWidth = 1;
          ctx.beginPath();
          ctx.moveTo(zoom.width / 2, 0); ctx.lineTo(zoom.width / 2, zoom.height);
          ctx.moveTo(0, zoom.height / 2); ctx.lineTo(zoom.width, zoom.height / 2);
          ctx.stroke();
          ctx.strokeStyle = "red"; ctx.lineWidth = 2;
          ctx.beginPath();
          ctx.arc(zoom.width / 2, zoom.height / 2, 9, 0, 6.283);
          ctx.stroke();
        }};
        img.onclick = e => {{
          const r = img.getBoundingClientRect();
          const dx = e.clientX - r.left, dy = e.clientY - r.top;
          const nx = dx * img.naturalWidth / r.width;
          const ny = dy * img.naturalHeight / r.height;
          drawZoom(nx, ny);
          if (MODE === "point") {{
            pt = [nx, ny];
            mk.style.left = dx + "px"; mk.style.top = dy + "px";
            mk.style.display = "block";
            st.textContent = "selected (" + Math.round(nx) + ", " + Math.round(ny)
                           + ") - check the zoom, click again to adjust, then Confirm";
            st.style.color = "#0a0";
            ok.disabled = false;
          }} else if (!c1 || box) {{
            // first corner (or restart after a completed box)
            c1 = {{dx: dx, dy: dy, nx: nx, ny: ny}};
            box = null;
            rect.style.display = "none";
            mk.style.left = dx + "px"; mk.style.top = dy + "px";
            mk.style.display = "block";
            st.textContent = "corner 1 set (" + Math.round(nx) + ", " + Math.round(ny)
                           + ") - now click the OPPOSITE corner";
            st.style.color = "#c60";
            ok.disabled = true;
          }} else {{
            box = [Math.min(c1.nx, nx), Math.min(c1.ny, ny),
                   Math.max(c1.nx, nx), Math.max(c1.ny, ny)];
            rect.style.left = Math.min(c1.dx, dx) + "px";
            rect.style.top = Math.min(c1.dy, dy) + "px";
            rect.style.width = Math.abs(dx - c1.dx) + "px";
            rect.style.height = Math.abs(dy - c1.dy) + "px";
            rect.style.display = "block";
            mk.style.display = "none";
            st.textContent = "box set (" + box.map(Math.round).join(", ")
                           + ") - click again to restart, or Confirm";
            st.style.color = "#0a0";
            ok.disabled = false;
          }}
        }};
        ok.onclick = () => {{
          img.style.border = "4px solid lime";
          st.textContent = "confirmed";
          resolve(MODE === "point" ? pt : box);
        }};
        document.getElementById("{uid}_skip").onclick = () => resolve(null);
      }})
    ''')
    return tuple(coords) if coords else None

def ask_click_local(image_path: Path, title: str, mode: str = "point"):
    print(f"{title}\nimage: {image_path}")
    if mode == "box":
        raw = input("enter 'x0,y0,x1,y1' box coords, or blank to skip: ").strip()
        return tuple(map(float, raw.split(","))) if raw else None
    raw = input("enter 'x,y' pixel coords, or blank to skip: ").strip()
    if not raw:
        return None
    x, y = raw.split(",")
    return (float(x), float(y))

clicks = load_clicks()
pending = [it for it in click_items
           if item_key(it) not in clicks or it["target_object"] in REDO_OBJECTS]
if REDO_OBJECTS:
    print(f"REDO_OBJECTS={sorted(REDO_OBJECTS)}: re-asking those even if answered")
print(f"already answered: {len(clicks)} | pending: {len(pending)}\n")

for i, item in enumerate(pending):
    image_path = resolve_click_image(item)
    if image_path is None:
        clicks[item_key(item)] = {**item, "point": None, "error": "no_frames_extracted"}
        save_clicks(clicks)
        print(f"auto-skip {item_key(item)}: no frames extracted")
        continue
    actual_idx = int(image_path.stem)  # may differ from the intended click frame
    mode = "box" if item["target_object"] in BOX_PROMPT_OBJECTS else "point"
    action = "draw a box around the WHOLE object (2 corner clicks)" if mode == "box" \
             else "click the object"
    title = (f"[{i + 1}/{len(pending)}] {item['target_object']} - "
             f"{item['sequence_id']} ({item['camera']} camera) - {action}")
    try:
        result = (ask_click_colab(image_path, title, mode) if IN_COLAB
                  else ask_click_local(image_path, title, mode))
    except Exception as e:
        clicks[item_key(item)] = {**item, "point": None, "error": f"{type(e).__name__}: {e}"}
        save_clicks(clicks)
        print(f"auto-skip {item_key(item)}: {type(e).__name__}: {e}")
        continue
    # store the frame index actually shown, so propagation seeds on the clicked frame
    entry = {**item, "click_frame_idx": actual_idx, "point": None, "box": None}
    if result is not None:
        if len(result) == 4:
            entry["box"] = list(result)
        else:
            entry["point"] = list(result)
    clicks[item_key(item)] = entry
    save_clicks(clicks)

n_answered = sum(1 for v in clicks.values() if v.get("point") or v.get("box"))
print(f"\nclick round done: {n_answered} annotated, "
      f"{len(clicks) - n_answered} skipped -> {CLICKS_JSON}")

## 6. Propagation round (GPU)

SAM3 tracker (video point-prompt mode of `facebook/sam3`): seed with the click on the
click frame and propagate through the extracted frames. Frames *before* the click frame
are labeled by running the reversed frame list through the same forward pass, so no
reverse-propagation API is needed. Output per sequence: mask -> bbox per frame.

If your transformers version exposes a different tracker API, check the model card at
`https://huggingface.co/facebook/sam3` and adjust the marked lines.

In [ ]:
from transformers import Sam3TrackerVideoModel, Sam3TrackerVideoProcessor

DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
tracker = Sam3TrackerVideoModel.from_pretrained(SAM3_MODEL_ID).to(DEVICE, dtype=DTYPE).eval()
tracker_processor = Sam3TrackerVideoProcessor.from_pretrained(SAM3_MODEL_ID)

def masks_to_bbox(mask: np.ndarray):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def load_frames(frames_dir: Path) -> list[Image.Image]:
    return [Image.open(p).convert("RGB") for p in sorted(frames_dir.glob("*.jpg"))]

@torch.inference_mode()
def track_forward(frames: list, seed_idx: int, prompt: dict) -> dict[int, np.ndarray]:
    """Seed at frames[seed_idx] with prompt {'point': (x, y)} or
    {'box': (x0, y0, x1, y1)}, propagate forward; returns {frame_idx: bool mask}."""
    h, w = frames[0].height, frames[0].width
    session = tracker_processor.init_video_session(                       # API line 1
        video=frames, inference_device=DEVICE, dtype=DTYPE,
    )
    if prompt.get("box"):
        seed_kwargs = dict(input_boxes=[[list(map(float, prompt["box"]))]])
    else:
        seed_kwargs = dict(input_points=[[[list(prompt["point"])]]],
                           input_labels=[[[1]]])
    tracker_processor.add_inputs_to_inference_session(                    # API line 2
        session, frame_idx=seed_idx, obj_ids=1, **seed_kwargs,
    )
    masks = {}
    for output in tracker.propagate_in_video_iterator(                    # API line 3
        session, start_frame_idx=seed_idx,
    ):
        video_masks = tracker_processor.post_process_masks(
            [output.pred_masks], original_sizes=[[h, w]], binarize=True,
        )[0]
        masks[output.frame_idx] = video_masks[0, 0].cpu().numpy().astype(bool)
    return masks

import gc
import time

def free_gpu():
    """Each tracker session keeps a per-video memory bank on the GPU; release it
    between sequences or memory accumulates until OOM (~sequence 90-100 on a T4)."""
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def propagate_sequence(entry: dict) -> list[dict]:
    frames_dir = Path(entry["frames_dir"])
    frames = load_frames(frames_dir)
    idx = entry["click_frame_idx"]
    prompt = {"point": entry.get("point"), "box": entry.get("box")}

    results = track_forward(frames, idx, prompt)
    free_gpu()  # drop the forward session's memory bank before the reverse pass
    if idx > 0:
        # frames before the click: same forward pass on the reversed list
        rev = track_forward(list(reversed(frames)), len(frames) - 1 - idx, prompt)
        for ridx, mask in rev.items():
            results.setdefault(len(frames) - 1 - ridx, mask)

    rows = []
    for frame_idx, mask in sorted(results.items()):
        bbox = masks_to_bbox(mask)
        rows.append({
            "sequence_id": entry["sequence_id"],
            "target_object": entry["target_object"],
            "camera": entry["camera"],
            "frame_idx": frame_idx,
            "frame_path": str(frames_dir / f"{frame_idx:05d}.jpg"),
            "x0": bbox[0] if bbox else None,
            "y0": bbox[1] if bbox else None,
            "x1": bbox[2] if bbox else None,
            "y1": bbox[3] if bbox else None,
            "mask_area_frac": float(mask.mean()),
        })
    return rows

BOXES_CSV = OUTPUT_DIR / "propagated_boxes.csv"
BOX_COLS = ["sequence_id", "target_object", "camera", "frame_idx", "frame_path",
            "x0", "y0", "x1", "y1", "mask_area_frac"]

def load_boxes_tolerant(path: Path) -> pd.DataFrame:
    """Read propagated_boxes.csv even when legacy 7-column rows (bbox_xyxy tuple)
    and current 10-column rows were appended into one file without a new header."""
    import csv as _csv
    rows = []
    with path.open(newline="") as f:
        for rec in _csv.reader(f):
            if not rec or rec[0] == "sequence_id":   # header lines (either schema)
                continue
            if len(rec) == len(BOX_COLS):            # current schema
                rows.append(dict(zip(BOX_COLS, rec)))
            elif len(rec) == 7:                      # legacy: bbox_xyxy at index 5
                nums = re.findall(r"-?\d+", rec[5] or "")
                bbox = list(map(int, nums)) if len(nums) == 4 else [None] * 4
                rows.append({"sequence_id": rec[0], "target_object": rec[1],
                             "camera": rec[2], "frame_idx": rec[3], "frame_path": rec[4],
                             "x0": bbox[0], "y0": bbox[1], "x1": bbox[2], "y1": bbox[3],
                             "mask_area_frac": rec[6]})
            # anything else: malformed line, drop it
    df = pd.DataFrame(rows, columns=BOX_COLS)
    for col in ["frame_idx", "x0", "y0", "x1", "y1", "mask_area_frac"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# Resume support: skip sequence/cameras already propagated in a previous (crashed) run.
# Mixed/legacy files are repaired in place (original kept as a timestamped backup).
done_keys: set[str] = set()
if BOXES_CSV.exists():
    try:
        prev = pd.read_csv(BOXES_CSV)
        needs_repair = "bbox_xyxy" in prev.columns
    except Exception:
        needs_repair = True
    if needs_repair:
        prev = load_boxes_tolerant(BOXES_CSV)
        backup = BOXES_CSV.with_name(
            f"propagated_boxes_backup_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        BOXES_CSV.rename(backup)
        prev.to_csv(BOXES_CSV, index=False)
        print(f"repaired mixed-schema CSV: {len(prev)} rows kept "
              f"(original saved as {backup.name})")
    if REDO_OBJECTS:
        redo_mask = prev["target_object"].isin(list(REDO_OBJECTS))
        if redo_mask.any():
            prev = prev[~redo_mask]
            prev.to_csv(BOXES_CSV, index=False)
            print(f"REDO_OBJECTS={sorted(REDO_OBJECTS)}: dropped {int(redo_mask.sum())} "
                  f"old rows so those sequences re-propagate with the new prompts")
    done_keys = set(prev["sequence_id"].astype(str) + "::" + prev["camera"].astype(str))
    print(f"resuming: {len(done_keys)} sequence/cameras already propagated - skipping those")

clicks = load_clicks()
clicked_entries = [v for v in clicks.values()
                   if (v.get("point") or v.get("box")) and item_key(v) not in done_keys]

for j, entry in enumerate(clicked_entries):
    rows = None
    for attempt in range(2):
        try:
            rows = propagate_sequence(entry)
            break
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on {entry['sequence_id']} - clearing GPU cache, retrying "
                  f"({attempt + 1}/2)")
            free_gpu()
        except Exception as e:
            print(f"FAILED {entry['sequence_id']}: {type(e).__name__}: {e}")
            break
    free_gpu()
    if not rows:
        continue
    # append immediately so a crash never loses completed sequences
    pd.DataFrame(rows).to_csv(BOXES_CSV, mode="a",
                              header=not BOXES_CSV.exists(), index=False)
    found = sum(1 for r in rows if r["x0"] is not None)
    print(f"[{j + 1}/{len(clicked_entries)}] {entry['sequence_id']} "
          f"({entry['target_object']}): {found}/{len(rows)} frames with mask")

boxes_df = pd.read_csv(BOXES_CSV) if BOXES_CSV.exists() else pd.DataFrame()
print(f"\n{len(boxes_df)} frame labels total -> {BOXES_CSV}")

## 7. Export YOLO labels + QA overlays

In [ ]:
LABELS_ROOT.mkdir(parents=True, exist_ok=True)
OVERLAYS_ROOT.mkdir(parents=True, exist_ok=True)

def row_bbox(row) -> tuple | None:
    """Bbox from either schema: x0..y1 columns, or the legacy 'bbox_xyxy' tuple
    string written by runs before the resume-capable version."""
    if pd.notna(row.get("x0")):
        return int(row["x0"]), int(row["y0"]), int(row["x1"]), int(row["y1"])
    nums = re.findall(r"-?\d+", str(row.get("bbox_xyxy", "") or ""))
    return tuple(map(int, nums)) if len(nums) == 4 else None

def resolve_frame_path(row) -> Path | None:
    """frame_path rows written in earlier sessions may point at a different
    OUTPUT_DIR; fall back to the current frames folder before giving up."""
    p = Path(str(row["frame_path"]))
    if p.exists():
        return p
    alt = FRAMES_ROOT / str(row["sequence_id"]) / str(row["camera"]) / p.name
    return alt if alt.exists() else None

n_labels = 0
missing_frames: list[str] = []
for _, row in boxes_df.iterrows():
    bbox = row_bbox(row)
    if bbox is None:
        continue
    x0, y0, x1, y1 = bbox
    frame_path = resolve_frame_path(row)
    img = cv2.imread(str(frame_path)) if frame_path else None
    if img is None:
        missing_frames.append(str(row["frame_path"]))
        continue
    h, w = img.shape[:2]
    cls = CLASS_IDS.get(row["target_object"], -1)
    cx, cy = (x0 + x1) / 2 / w, (y0 + y1) / 2 / h
    bw, bh = (x1 - x0) / w, (y1 - y0) / h

    stem = f"{row['sequence_id']}_{row['camera']}_{row['frame_idx']:05d}"
    (LABELS_ROOT / f"{stem}.txt").write_text(
        f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"
    )
    n_labels += 1

    if row["frame_idx"] % 5 == 0:  # QA overlay every 5th frame
        cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)
        cv2.putText(img, row["target_object"], (x0, max(y0 - 6, 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.imwrite(str(OVERLAYS_ROOT / f"{stem}.jpg"), img)

print(f"YOLO labels written: {n_labels} -> {LABELS_ROOT}")
print(f"QA overlays        : {OVERLAYS_ROOT}")
if missing_frames:
    pd.Series(sorted(set(missing_frames))).to_csv(
        OUTPUT_DIR / "missing_frames.csv", index=False, header=["frame_path"])
    print(f"skipped {len(missing_frames)} rows whose frame images no longer exist "
          f"-> {OUTPUT_DIR / 'missing_frames.csv'}")
    print("(frames from an older session/output dir - re-run the frame extraction "
          "cell, then re-run this cell to pick them up)")

In [ ]:
# Session summary + a QA gallery sample
if len(boxes_df):
    summary = (
        boxes_df.assign(has_box=boxes_df.apply(lambda r: row_bbox(r) is not None, axis=1))
        .groupby("target_object")
        .agg(sequences=("sequence_id", "nunique"),
             frames=("has_box", "size"),
             frames_with_box=("has_box", "sum"))
    )
    summary["labels_per_click"] = (summary["frames_with_box"] / summary["sequences"]).round(1)
    display(summary)

    sample = sorted(OVERLAYS_ROOT.glob("*.jpg"))
    sample = random.Random(SEED).sample(sample, min(9, len(sample)))
    fig, axes = plt.subplots(3, 3, figsize=(15, 9))
    for ax in axes.flat:
        ax.axis("off")
    for ax, p in zip(axes.flat, sample):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.stem, fontsize=7)
    plt.show()

## Next steps

1. Eyeball the QA gallery: propagated boxes should stay glued to the object as it
   grows during the approach. Drifting boxes -> re-click that sequence (delete its
   entry from `clicks.json` and rerun the click round).
2. Merge these labels with the text-prompted SAM3 labels for the passing classes into
   one YOLO dataset (split by sequence!).
3. Reserve a slice of the clicked sequences as the human-verified test set - do NOT
   train on those.